# Data ingestion Layer:
- get documents
- create chunks
- embeddings
- call the llm transformer instance to generate our knowledge graph(KG)

#### **Load Libraries**

In [14]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters.character import RecursiveCharacterTextSplitter
import ollama
import re # regex - regular experessions 
from langchain_ollama import ChatOllama
from langchain_neo4j import Neo4jGraph, LLMGraphTransformer

#### **Load Data**

In [3]:
loader = PyPDFLoader("data/Darwinism.pdf")
pages = loader.load()

In [4]:
pages

[Document(metadata={'producer': 'Skia/PDF m154', 'creator': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) HeadlessChrome/154.0.0.0 Safari/537.36', 'creationdate': '2026-10-16T07:09:35+00:00', 'title': 'Darwinism - Wikipedia', 'moddate': '2026-10-16T07:09:35+00:00', 'source': 'data/Darwinism.pdf', 'total_pages': 9, 'page': 0, 'page_label': '1'}, page_content='Charles Darwin in 1868\nDarwinism\nDarwinism is a term used to describe a theory of biological\nevolution developed by the English naturalist Charles Darwin\n(1809–1882) and his contemporaries. The theory states that\nall species of organisms arise and develop through the natural\nselection of small, inherited variations that increase the\nindividual\'s ability to compete, survive, and reproduce. Also\ncalled Darwinian theory, it originally included the broad\nconcepts of transmutation of species or of evolution which\ngained general scientific acceptance after Darwin published\nOn the Origin of Species in

In [5]:
# clean data
def cleanText(text):
    text = re.sub(r'-\n(\w)', r'\1', text)     # re-join words hyphenated across lines
    text = text.replace('•', '\n- ')           # bullets as their own lines
    text = re.sub(r'[ \t]+', ' ', text)        # collapse spaces/tabs only
    text = re.sub(r'\n{3,}', '\n\n', text)     # cap blank lines
    return text.strip()

#### **Split and create chunks**

In [10]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 800, chunk_overlap= 80) # created an instance of the text splitter
chunks = splitter.split_documents(pages)

In [12]:
# embedding function
def createEmbedding(text):
    response = ollama.embed(
    model='nomic-embed-text-v2-moe',
    input=text,
    )
    return response.embeddings[0]

In [13]:
doc_embeddings = [] # this is our list of embeddings from out document

for chunk in chunks:
    chunk_embedding = createEmbedding(chunk.page_content)
    doc_embeddings.append(chunk_embedding) 

In [15]:
# creates an instance of the model we will use low temperature and high content window
deepseek_llm = ChatOllama(model="deepseek-r1:1.5b", temperature=0)
mistral_llm =  ChatOllama(model="mistral:7b", temperature=0)

In [17]:
# connect to neo4J DB
graph = Neo4jGraph(url="neo4j://127.0.0.1:7687", username="neo4j", password="neo4j123")

# transformer init
deep_seek_transformer = LLMGraphTransformer(llm=deepseek_llm)
mistral_transformer = LLMGraphTransformer(llm=mistral_llm)

#### **create knowledge graph**

In [18]:
# graph_documents = transformer.convert_to_graph_documents([chunks[0]]) 
graph_documents = []
for chunk in chunks:
    graph_documents.append(deep_seek_transformer.convert_to_graph_documents([chunk]))

KeyboardInterrupt: 

In [19]:
graph_documents

[[GraphDocument(nodes=[Node(id='Person', type='Person', properties={}), Node(id='Darwinism', type='Theory', properties={})], relationships=[Relationship(source=Node(id='Person', type='Person', properties={}), target=Node(id='Theory', type='Theory', properties={}), type='IS', properties={})], source=Document(metadata={'producer': 'Skia/PDF m154', 'creator': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) HeadlessChrome/154.0.0.0 Safari/537.36', 'creationdate': '2026-10-16T07:09:35+00:00', 'title': 'Darwinism - Wikipedia', 'moddate': '2026-10-16T07:09:35+00:00', 'source': 'data/Darwinism.pdf', 'total_pages': 9, 'page': 0, 'page_label': '1'}, page_content="Charles Darwin in 1868\nDarwinism\nDarwinism is a term used to describe a theory of biological\nevolution developed by the English naturalist Charles Darwin\n(1809–1882) and his contemporaries. The theory states that\nall species of organisms arise and develop through the natural\nselection of small, inherited va